# UNCP 4 | Model and executed results

کد مدل و یک بررسی اجرایی در همین دفترچه اجرا شده‌اند. اعداد سه seed از فایل‌های نتیجهٔ آزمایش کامل که پیش‌تر اجرا شده‌اند، در سلول آخر ثبت و با میانگین همان فایل‌ها کنترل می‌شوند. این دفترچه آموزش کامل سه seed را دوباره اجرا نمی‌کند؛ برای بازآموزی، داده، کد benchmark و تنظیمات همان پروژه لازم است.

In [1]:
import torch
from torch import nn
import torch.nn.functional as F

"""UNCP 4: typed graph attention, GRU, transition graph, adaptive fusion.

The candidate mask is a conservative observed-page proxy. It is built only
from a supplied UI inventory and training-fold transitions, never target pages.
"""
import torch
from torch import nn
import torch.nn.functional as F


class TypedGraphAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.query = nn.ModuleList([nn.Linear(dim, dim) for _ in range(2)])
        self.key = nn.ModuleList([nn.Linear(dim, dim) for _ in range(2)])
        self.value = nn.ModuleList([nn.Linear(dim, dim) for _ in range(2)])
        self.fusion = nn.Linear(3 * dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, nodes, relations):
        channels = [nodes]
        for i, edges in enumerate(relations):
            scores = self.query[i](nodes) @ self.key[i](nodes).T / nodes.shape[-1] ** .5
            weights = scores.masked_fill(~edges, -1e9).softmax(-1)
            channels.append(weights @ self.value[i](nodes))
        return self.norm(nodes + F.gelu(self.fusion(torch.cat(channels, -1))))


class UNCP4NextClick(nn.Module):
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        dim, hidden = config['embedding_dim'], config['hidden_dim']
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.page_embedding = nn.Embedding(config['n_pages'], dim)
        self.graph = TypedGraphAttention(dim)
        self.gru = nn.GRU(dim, hidden, batch_first=True)
        self.graph_projection = nn.Linear(dim, hidden)
        self.transition_projection = nn.Linear(n_classes, hidden)
        self.gate = nn.Linear(3 * hidden, 3)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(config['dropout'])
        self.output = nn.Linear(hidden, n_classes)
        self.register_buffer('page_edges', config['page_edges'].bool())
        self.register_buffer('transition_edges', config['transition_edges'].bool())
        self.register_buffer('transition_probs', config['transition_probs'].float())
        self.register_buffer('candidate_mask', config['candidate_mask'].bool())

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        batch = torch.arange(x.shape[0], device=x.device)
        nodes = torch.cat([self.embedding.weight, self.page_embedding.weight], 0)
        graph_nodes = self.graph(nodes, (self.page_edges, self.transition_edges))
        graph_tokens = graph_nodes[x]
        recurrent, _ = self.gru(self.embedding(x) + graph_tokens)
        recurrent = recurrent[batch, lengths - 1]
        observed = x.ne(0).float().unsqueeze(-1)
        graph_state = self.graph_projection((graph_tokens * observed).sum(1) / observed.sum(1).clamp_min(1))
        last = x[batch, lengths - 1]
        transition_state = self.transition_projection(self.transition_probs[last])
        states = torch.stack([recurrent, graph_state, transition_state], 1)
        weights = torch.softmax(self.gate(torch.cat([recurrent, graph_state, transition_state], -1)), -1)
        logits = self.output(self.dropout(self.norm((states * weights.unsqueeze(-1)).sum(1))))
        return logits.masked_fill(~self.candidate_mask[last], -1e9)


In [2]:
cfg=dict(embedding_dim=32,hidden_dim=64,dropout=.2,n_pages=2,page_edges=torch.eye(12,dtype=torch.bool),transition_edges=torch.eye(12,dtype=torch.bool),transition_probs=torch.full((10,5),.2),candidate_mask=torch.ones(10,5,dtype=torch.bool))
model=UNCP4NextClick(10,5,cfg)
x=torch.tensor([[2,2,3,0],[4,5,6,7]])
lengths=torch.tensor([3,4])
out=model(x,lengths)
assert out.shape==(2,5) and torch.isfinite(out).all()
print('UNCP 4 forward pass OK:',tuple(out.shape))

UNCP 4 forward pass OK: (2, 5)


## Recorded experiment results

The following cell executes a consistency check over the recorded three-seed result rows and displays them. It does not retrain the model.

In [3]:
import json, numpy as np
record = json.loads("{\"summary\": {\"protocol\": \"grouped_user\", \"model\": \"uncp4\", \"seeds\": \"3\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.5984491830517862\", \"top3\": \"0.8107941782824087\", \"macro_target_top1\": \"0.5156944881594836\", \"macro_target_top3\": \"0.6801169307784917\", \"macro_user_top1\": \"0.6107045989967255\", \"macro_user_top3\": \"0.818546311470632\", \"cluster_ci95_top1_low\": \"0.5487152094633457\", \"cluster_ci95_top1_high\": \"0.6477397189779494\", \"cluster_ci95_top3_low\": \"0.7803692616815592\", \"cluster_ci95_top3_high\": \"0.8358363308189471\", \"top1_seed_sd\": \"0.013969118388647223\", \"top3_seed_sd\": \"0.008787135234241824\", \"macro_target_top1_seed_sd\": \"0.017129077511917827\", \"macro_target_top3_seed_sd\": \"0.01779798043601871\", \"macro_user_top1_seed_sd\": \"0.013848603369107029\", \"macro_user_top3_seed_sd\": \"0.00667036035667742\"}, \"runs\": [{\"protocol\": \"grouped_user\", \"model\": \"uncp4\", \"seed\": \"11\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.5938336564201976\", \"top3\": \"0.8091941290501247\", \"macro_target_top1\": \"0.5118135455238988\", \"macro_target_top3\": \"0.6665667087872567\", \"macro_user_top1\": \"0.5998210975130807\", \"macro_user_top3\": \"0.8167574876752866\", \"best_epoch\": \"21\", \"epochs_run\": \"26\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp4\", \"seed\": \"23\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.5873719191359734\", \"top3\": \"0.802917012831164\", \"macro_target_top1\": \"0.5008388584534144\", \"macro_target_top3\": \"0.6735106472058255\", \"macro_user_top1\": \"0.6060001257184275\", \"macro_user_top3\": \"0.8129527506988308\", \"best_epoch\": \"19\", \"epochs_run\": \"24\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp4\", \"seed\": \"37\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6141419735991877\", \"top3\": \"0.8202713929659374\", \"macro_target_top1\": \"0.5344310605011375\", \"macro_target_top3\": \"0.7002734363423929\", \"macro_user_top1\": \"0.6262925737586683\", \"macro_user_top3\": \"0.8259286960377783\", \"best_epoch\": \"29\", \"epochs_run\": \"30\"}], \"behavior\": null}")
summary, runs = record['summary'], record['runs']
top1 = float(summary.get('test_mean_top1', summary.get('top1')))
top3 = float(summary.get('test_mean_top3', summary.get('top3')))
seed_top1 = np.array([float(row['top1']) for row in runs])
seed_top3 = np.array([float(row['top3']) for row in runs])
assert np.isclose(seed_top1.mean(), top1, atol=1e-9)
assert np.isclose(seed_top3.mean(), top3, atol=1e-9)
print('Recorded grouped-user experiment: 10,833 test targets, 11 held-out users')
print(f'UNCP 4  Top-1: {100*top1:.2f}%  Top-3: {100*top3:.2f}%')
print('Seeds 11, 23, 37 Top-3:', [round(100*x,2) for x in seed_top3])
if record['behavior']:
    for group in record['behavior']:
        print(group['group'], 'n=',group['n'],'UNCP6 Top-3=',round(100*group['uncp6_top3'],2),'UNCP5 Top-3=',round(100*group['uncp5_top3'],2))

Recorded grouped-user experiment: 10,833 test targets, 11 held-out users
UNCP 4  Top-1: 59.84%  Top-3: 81.08%
Seeds 11, 23, 37 Top-3: [np.float64(80.92), np.float64(80.29), np.float64(82.03)]
